# 15 — open-appsec: real təmiz veb trafikində yanlış bloklama

**Sual:** detektorlar real saytların təmiz HTTP trafikində nə qədər yanlış bloklama verir?

- Məlumat: open-appsec WAF Comparison Project (`openappsec/waf-comparison-project`, Apache 2.0). Repoda `Data` qovluğu yoxdur;
  `config.py` təmiz dəsti `legitimate.zip`-dən yükləyir: 185 real saytdan brauzerlə yazılmış **1 040 242 HTTP sorğusu**, hamısı təmiz.
  Yalnız bu (legitimate) dəst istifadə olunur.
- Modellərimiz tək parametr dəyəri üzərində öyrədilib. Ona görə hər sorğudan middleware-in (`middleware/src/sqliGuard.js`) yoxlayacağı dəyərlər çıxarılır
  (`src/openappsec.py`): URL query (Express kimi bir dəfə decode), body — `application/json` (bütün sətir yarpaqları), `x-www-form-urlencoded`, `multipart/form-data` (fayl olmayan sahələr);
  digər body növləri (text/plain, protobuf, binar beacon-lar, CSP hesabatları) Express parserləri tərəfindən oxunmur və yoxlanılmır. Açarlar yox, yalnız dəyərlər; 4 096 simvoldan uzun dəyərlər kəsilir (middleware standartı).
- Detektorlar (seed 42–46): TF-IDF LR / SVM / RF / NB baza, LR + normallaşdırma, LR + normallaşdırma + augmentation (12/14 ilə eyni modellər), char-CNN baza və normallaşdırma (13 ilə eyni),
  qara siyahı ×2 və libinjection (deterministik, seed-dən asılı deyil).
- **Dəyər səviyyəsində FPR:** bloklanan unikal dəyərlərin payı (və tezliklə çəkili pay).
  **Sorğu səviyyəsində FPR:** sorğunun dəyərlərindən biri bloklanırsa, sorğu bloklanmış sayılır; məxrəc — ən azı bir dəyəri olan sorğular (dəyərsiz sorğu bloklana bilməz; bütün sorğulara görə pay da verilir).

In [ ]:
import sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import openappsec as oa
from charcnn import CONFIGS as CNN_CONFIGS, get_cnn
from detectors import BL_INPUT, BL_SEARCH, LIBINJ, train_all
from experiments import load_clean, make_split

RES, FIG = ROOT / "results", ROOT / "figures" / "paper"
FIG.mkdir(parents=True, exist_ok=True)
SEEDS = [42, 43, 44, 45, 46]
DEPLOYED = "LR + norm. + augmentation"   # middleware modeli
clean = load_clean()
oa.fetch()
requests, values = oa.load()
print(len(requests), "sorğu,", len(values), "dəyər")

## 1. Dəyərlərin çıxarılması

In [ ]:
codes, uniq = pd.factorize(values.value)
values["code"] = codes
n_req = len(requests)
has_val = requests.n_values.to_numpy() > 0
summary = pd.Series({
    "sorğular": n_req, "saytlar (fayllar)": requests.site.nunique(),
    "ən azı bir dəyəri olan sorğular": int(has_val.sum()),
    "dəyərlər (hamısı)": len(values), "unikal dəyərlər": len(uniq),
    "boş dəyərlər (hamısı içində, %)": round(100 * (values.value == "").mean(), 2),
    "4096-dan uzun (kəsilmiş) dəyərlər": int(values.truncated.sum()),
    **{f"dəyərlər: {k}": int(v) for k, v in values.source.value_counts().items()},
    "median dəyər uzunluğu": float(pd.Series(uniq).str.len().median()),
})
summary.to_csv(RES / "15_extraction_summary.csv", header=["value"])
display(summary.to_frame("dəyər"))
requests.content_type.value_counts().head(15)

## 2. FPR — 5 seed

In [ ]:
U = pd.Series(uniq)
occ = np.bincount(codes, minlength=len(uniq))           # hər unikal dəyərin tezliyi
req_of_val = values.req_id.to_numpy()

def chunked(det, xs, n=50_000):
    return np.concatenate([np.asarray(det(xs[i:i + n])) for i in range(0, len(xs), n)]).astype(np.int8)

def fpr_rows(name, seed, pred_u):
    pred_occ = pred_u[codes]
    blocked_req = np.bincount(req_of_val, weights=pred_occ, minlength=n_req) > 0
    return [{"seed": seed, "detector": name, "metric": m, "value": v} for m, v in (
        ("fpr_value_unique", pred_u.mean()),
        ("fpr_value_weighted", pred_occ.mean()),
        ("fpr_request", blocked_req[has_val].mean()),
        ("fpr_request_all", blocked_req.mean()),
        ("blocked_requests", blocked_req.sum()))]

rows, preds42 = [], {}
xs = U.tolist()
for name, det in (("Qara siyahı (sanitizeInput)", BL_INPUT), ("Qara siyahı (sanitizeSearch)", BL_SEARCH),
                  ("libinjection 4.0.1", LIBINJ)):
    t0 = time.perf_counter()
    p = chunked(det, xs)
    preds42[name] = p
    for s in SEEDS:                                     # deterministik: hər seed üçün eyni
        rows += fpr_rows(name, s, p)
    print(f"{name:30s} {time.perf_counter() - t0:.0f} s")

for s in SEEDS:
    t0 = time.perf_counter()
    train, val, _ = make_split(clean, s)
    dets = {k: m.predict for k, m in train_all(s, train, val).items()}
    for cfg in CNN_CONFIGS:
        dets[cfg] = get_cnn(cfg, s, train, val).predict
    for name, det in dets.items():
        p = chunked(det, xs)
        rows += fpr_rows(name, s, p)
        if s == 42:
            preds42[name] = p
    print(f"seed {s}: {time.perf_counter() - t0:.0f} s")

raw = pd.DataFrame(rows)
raw.to_csv(RES / "15_fpr_raw.csv", index=False)

In [ ]:
ORDER = ["LR baza", "SVM baza", "RF baza", "NB baza", "LR + normallaşdırma", "LR + norm. + augmentation",
         "CNN baza", "CNN + normallaşdırma", "Qara siyahı (sanitizeInput)", "Qara siyahı (sanitizeSearch)", "libinjection 4.0.1"]
pct = raw[raw.metric != "blocked_requests"]
a = pct.groupby(["detector", "metric"]).value.agg(["mean", "std"]) * 100
a.round(4).to_csv(RES / "15_fpr_mean_std.csv")
pm = lambda d: d["mean"].map("{:.3f}".format) + " ± " + d["std"].map("{:.3f}".format)
table = pm(a).unstack("metric").loc[ORDER, ["fpr_value_unique", "fpr_value_weighted", "fpr_request", "fpr_request_all"]]
blocked = raw[raw.metric == "blocked_requests"].groupby("detector").value.mean().loc[ORDER].round(0).astype(int)
table["bloklanan sorğu (orta)"] = blocked
table.to_csv(RES / "15_fpr_table.csv")
table

## 3. Ən çox yanlış bloklanan 20 təmiz dəyər
Middleware modeli (LR + normallaşdırma + augmentation, seed 42) üçün — bloklanan sorğu sayına görə. Yanında digər detektorların (seed 42) eyni dəyəri bloklayıb-bloklamadığı və kobud səbəb qrupu
(`src/openappsec.py::reason`, prioritet sırası ilə ilk uyğun qayda).

In [ ]:
req_count = values.drop_duplicates(["req_id", "code"]).code.value_counts().reindex(range(len(uniq)), fill_value=0).to_numpy()
def top_fp(det_name, k=20):
    fp = np.flatnonzero(preds42[det_name] == 1)
    fp = fp[np.argsort(-req_count[fp], kind="stable")][:k]
    t = pd.DataFrame({"value": U.iloc[fp].str[:120].values, "bloklanan_sorğu": req_count[fp], "tezlik": occ[fp],
                      "səbəb": U.iloc[fp].map(oa.reason).values})
    t["digər detektorlar da bloklayır"] = [", ".join(n for n in ORDER if n != det_name and preds42[n][i]) for i in fp]
    return t

top20 = top_fp(DEPLOYED)
top20.to_csv(RES / "15_top20_false_blocks_lr_norm_aug_seed42.csv", index=False)
pd.set_option("display.max_colwidth", 120)
top20

In [ ]:
top20_base = top_fp("LR baza")
top20_base.to_csv(RES / "15_top20_false_blocks_lr_base_seed42.csv", index=False)
top20_base

In [ ]:
# Səbəb qrupları: hər detektorun bütün yanlış bloklanan unikal dəyərləri üzrə (seed 42), bloklanan sorğu sayı ilə çəkili
rs = U.map(oa.reason).to_numpy()
rows = []
for name in ORDER:
    fp = preds42[name] == 1
    s = pd.Series(req_count[fp]).groupby(rs[fp]).sum()
    for r, v in s.items():
        rows.append({"detector": name, "səbəb": r, "unikal_dəyər": int((rs[fp] == r).sum()), "bloklanan_sorğu": int(v)})
reasons = pd.DataFrame(rows)
reasons.to_csv(RES / "15_fp_reasons_seed42.csv", index=False)
reasons.pivot_table(index="səbəb", columns="detector", values="bloklanan_sorğu", fill_value=0)[ORDER]

In [ ]:
# Sayt üzrə sorğu səviyyəsində FPR (seed 42) — yanlış bloklamalar bir neçə saytda toplanırmı?
site_of_req = requests.site.to_numpy()
rows = []
for name in ORDER:
    blocked_req = np.bincount(req_of_val, weights=preds42[name][codes], minlength=n_req) > 0
    s = pd.Series(blocked_req[has_val]).groupby(site_of_req[has_val]).mean()
    rows.append({"detector": name, "saytlar": len(s), "FPR>0 olan saytlar": int((s > 0).sum()),
                 "FPR>1% olan saytlar": int((s > 0.01).sum()), "max sayt FPR %": round(100 * s.max(), 2),
                 "median sayt FPR %": round(100 * s.median(), 3)})
by_site = pd.DataFrame(rows).set_index("detector")
by_site.to_csv(RES / "15_fpr_by_site_seed42.csv")
by_site

In [ ]:
labels = [n.replace(" (sanitizeInput)", "\n(Input)").replace(" (sanitizeSearch)", "\n(Search)").replace("normallaşdırma", "norm.")
          .replace(" + augmentation", "\n+ aug.").replace("libinjection 4.0.1", "libinjection") for n in ORDER]
fig, ax = plt.subplots(figsize=(12, 4.2))
x, w = np.arange(len(ORDER)), 0.38
for i, (m, lab, col) in enumerate([("fpr_value_unique", "Dəyər səviyyəsi (unikal dəyərlər)", "#4a6fa5"),
                                    ("fpr_request", "Sorğu səviyyəsi (≥1 dəyəri olan sorğular)", "#d08c3f")]):
    d = a.xs(m, level="metric").loc[ORDER]
    b = ax.bar(x + (i - 0.5) * w, d["mean"], w, yerr=d["std"], capsize=2, color=col, label=lab)
    ax.bar_label(b, fmt="%.2f", fontsize=6, padding=2)
ax.set_yscale("symlog", linthresh=0.1)
ax.set_xticks(x, labels, fontsize=7)
ax.set_ylabel("FPR, % (orta ± std, 5 seed; simlog)")
ax.legend(fontsize=8, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False)
fig.savefig(FIG / "15_openappsec_fpr.png", dpi=300, bbox_inches="tight")
plt.show()